# Search for Overflow

## Kas Knicely, University of Alaska Fairbanks

This notebook searches Sentinel 1 SAR imagery for possible overflow locations along the Tanana River between Fairbanks and Nanana. This was created in support of the CCREL Arctic Trafficability Project. 

This was created with radiometrically terrain corrected (RTC) SAR imagery in mind, though - strictly speaking - non-RTC SAR imagery can be used as well. The non-RTC SAR imagery will only work in this tool <b>if</b> it was taken by the same sensor from the same look angle and has pixel locations that match each other. This is because a difference calculation is used to locate the possible overflow. 

To incorporate SAR imagery from MULTIPLE sensor sources (e.g., Sentinel 1 and plane-mounted), the data <b>must be</b> RTC. 

Approximate Date of Creation: 2026 Jan. 20

***
# 1. Load Python Libraries

In [ ]:
# Confirmed Necessary
import xarray as xr
from pathlib import Path
import pandas as pd
import glob
import re
import os
import rasterio
import matplotlib.pyplot as plt
# Possibly Necessary
import rioxarray as rxr
import time
import numpy as np
import time


***
# 2. Load Sentinel 1 SAR Imagery

## 2.1 Select folder from which to load S1 SAR Imagery

The below cell is a small bit of code to install the ipyfilechooser if necessary. 

In [ ]:
!pip install ipyfilechooser

This should be the folder containing the subsetted imagery of your region of interest. These files should be in a folder named 'RTC_GAMMA'. 

For example: <br>
Subsets were placed in a folder named 'subsets'. This folder should contain a folder named 'RTC_GAMMA' which will contain your subsetted tiffs. To run the below code, you will select the folder 'subsets'. 

In [ ]:
from ipyfilechooser import FileChooser
fc = FileChooser(Path.cwd())
display(fc)

## 2.2 Load S1 SAR Imagery

In [ ]:
### --- Get Dates from filenames --- ###
def get_dates(flnms):
    dates = []

    for flnm in flnms:
        date_regex = r'\d{8}'
        date = re.search(date_regex, str(flnm))
        if date:
            dates.append(date.group(0))
    return dates


### --- Load Geotiffs Function --- ###
def load_tiffs(parent, type_file, pola, stop_ind = -1):

    # Load the appropriate files
    if type_file == 'SAR':
        folder = parent+'RTC_GAMMA/'
        prefix = f'*{pola}'
    else:
        folder = parent+'Water_Masks/'
        prefix = '*combined'

    # Gather names of files corresponding to the file type and polarization we want
    tiff_dir = Path(folder)
    tiffs = [f for f in os.listdir(tiff_dir) if pola in f]
    
    # Gather the date of each file
    times = get_dates(tiffs)

    # Make 'tiffs' have full path
    tiffs = [Path(folder,tiff) for tiff in tiffs]
    
    # Create a list of indices based on the sorted order of times
    sorted_indices = sorted(range(len(times)), key=lambda i: times[i])
    
    # Sort the paths based on the times
    tiffs = [tiffs[i] for i in sorted_indices]
    
    # Sort the times axisdd
    times.sort()
    times = pd.DatetimeIndex(times)
    times.name = "time"
    
    # Create the dataset gathering the input images
    if stop_ind == -1:
        stop_ind = len(tiffs)
    da = xr.concat([rxr.open_rasterio(Path(tiff_dir,f)).squeeze(dim='band') for f in tiffs[:stop_ind]], dim=times[:stop_ind])
    da = da.drop_vars(['band', 'spatial_ref'])

    print(f"Dataset size in memory: {da.nbytes / 1e6:.2f} MB")

    return da, tiffs, times

In [ ]:
### --- Load data --- ###
# da_tot - Xarray containing all of the data. 
# tiffPaths_tot - posix paths array containing paths to all data files. 
# times_tot - list containing all of the dates of each data (note: this will also be in 'da_tot').

type_file = 'SAR'

# # Load VV tiffs
pola = 'VV' 
da_VV_tot, tiffPaths_VV_tot, times_VV_tot = load_tiffs(fc.selected, type_file, pola, stop_ind = -1)

# Load VH tiffs
pola = 'VH'
da_VH_tot, tiffPaths_VH_tot, times_VH_tot = load_tiffs(fc.selected, type_file, pola, stop_ind = -1)

Remove fill values. 

For data in decibels, this was found to be the smallest value. 

In [ ]:
# Replace fill values with NaN. 
myMin = np.min(da_VH_tot[0].values)
da_VH_tot = da_VH_tot.where(da_VH_tot != myMin, other=np.nan)
myMin = np.min(da_VV_tot[0].values)
da_VV_tot = da_VV_tot.where(da_VV_tot != myMin, other=np.nan)

In [ ]:
plt.figure()
plt.hist(da_VH_tot[0].values.flatten())
plt.show()

In [ ]:
plt.figure()
plt.hist(da_VV_tot[0].values.flatten())
plt.show()

***
# 3. Examine SAR Imagery

This section simply plots the raw SAR imagery. Data is assumed to be in decibels. 

In [ ]:
def subplot_SAR_in_dB(da_tot, vmin=-25, vmax=-5): 

    num_modes = len(da_tot)

    # Create subplots
    n_cols = 2
    n_rows = int(np.ceil(num_modes/2))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12,n_rows*4))

    # Flatten axes array for easier indexing. 
    axes = axes.flatten()

    # Loop through and plot the modes. 
    for i in range(num_modes): 
        ax = axes[i]
        # myData_dB = 20 * np.log10(np.abs(da_tot[i].copy(deep=True).values + 1e-10))
        # ax.imshow(myData_dB, cmap='gray_r', vmin=vmin, vmax=vmax, interpolation=None)
        # ax.imshow(abs(da_tot[i].values), cmap='gray', vmin=vmin, vmax=vmax, interpolation=None)
        ax.imshow(da_tot[i].values, cmap='gray', vmin=vmin, vmax=vmax, interpolation=None)
        ax.set_title(f"Date: {da_tot[i].time.values.astype('datetime64[D]')}")

    # Turn off unused subplots
    for i in range(num_modes, len(axes)): 
        ax = axes[i]
        ax.set_axis_off()

    # Adjust layout
    plt.tight_layout()

    # Show plot. 
    plt.show()
    
    return

In [ ]:
subplot_SAR_in_dB(da_VH_tot, vmin=-35, vmax=-5)

In [ ]:
subplot_SAR_in_dB(da_VV_tot, vmin=-30, vmax=-5)

In [ ]:
myMin = np.nanmin(da_VH_tot.values.flatten())
myMax = np.nanmax(da_VH_tot.values.flatten())

subplot_SAR_in_dB(da_VH_tot, 
                  vmin=myMin, 
                  vmax=myMax)

In [ ]:
myMin = np.nanmin(da_VV_tot.values.flatten())
myMax = np.nanmax(da_VV_tot.values.flatten())

subplot_SAR_in_dB(da_VV_tot, 
                  vmin=myMin, 
                  vmax=myMax)

***
# 4. Calculate Change in dB levels

This section has two subsections. 
* 4.1 - In this section, the change in decibels is calculated.
* 4.2 - In this section, some statistics on the data are displayed. 

## 4.1 Do the calculation

In [ ]:
# Initialize array to contain differences. 
t, y, x = da_VH_tot.shape[0]-1, da_VH_tot.shape[1], da_VH_tot.shape[2]
da_diffs_VH = np.zeros((t,y,x))
t, y, x = da_VV_tot.shape[0]-1, da_VV_tot.shape[1], da_VV_tot.shape[2]
da_diffs_VV = np.zeros((t,y,x))

In [ ]:
### --- Set up window sizes for search through SAR imagery --- ###

window_sizes = [9] # window sizes; must be odd value.
for i in range(len(window_sizes)): 
    if window_sizes[i] % 2 != 0:
        continue
    else: 
        print("WARNING!!!\nWARNING!!!\nWARNING!!!\nWindow sizes must be odd. Increasing value by 1.")
        window_sizes[i] = window_sizes[i]+1

# remove duplicates. 
window_sizes = list(set(window_sizes))
window_sizes.sort()

In [ ]:
from scipy import signal

def daDiffCalc(da_tot, w_s):
    # Function to get the difference between consecutive days in a data array. 
    # Note: This uses a median filter to stabilize values. 
    
    # da_tot - xarray data to get difference between consecutive days. 
    # w_s - window size to consider. 
    
    ts, ys, xs = da_tot.shape
    da_diffs = np.zeros((ts-1,ys,xs))
    temp = np.zeros((ys, xs))

    for i in range(ts-1): 
        temp = da_tot[i+1].values - da_tot[i].values
        da_diffs[i] = signal.medfilt2d(temp, kernel_size=w_s)

    return da_diffs

In [ ]:
### --- Calculate difference between consecutive days --- ###

# Initialize container for differences. 
ts, ys, xs = da_VH_tot.shape
da_diffs_VH = np.zeros((len(window_sizes),ts-1,ys,xs))
da_diffs_VV = np.zeros((len(window_sizes),ts-1,ys,xs))

# Calculate differences. 
start_time = time.time()
for i, w_s in enumerate(window_sizes):
    # Run function to get differences. 
    da_diffs_VH[i] = daDiffCalc(da_VH_tot, w_s)
    da_diffs_VV[i] = daDiffCalc(da_VV_tot, w_s)

end_time = time.time()

print(f'It took {end_time-start_time:.2f} seconds to calculate decibel changes between consecutive days. ')

## 4.2 Difference Statistics

Plot some info about the differences calculated. 

The first plot shows the mean, mean +- standard deviation, and median for: all of the data, only the positive values, and only the negative values. This is useful for knowing where the bulk of the values (and therefore NOT overflow) is. 

The second plot shows the histograms of the differences for each date pair. This is useful for determining the threshold used in the next step. Overflow generally shows a full decibel drop in signal. 

In [ ]:
# statistics (mean, median, standard deviation) plotting. 
def diffPlotStats(da_diffs, da_tot): 

    num_diffs = len(da_diffs)

    # Initialize containers for stats. 
    diffs_mean_all = np.zeros((num_diffs, 1))
    diffs_mean_pos = np.zeros((num_diffs, 1))
    diffs_mean_neg = np.zeros((num_diffs, 1))
    diffs_med_all  = np.zeros((num_diffs, 1))
    diffs_med_pos  = np.zeros((num_diffs, 1))
    diffs_med_neg  = np.zeros((num_diffs, 1))
    diffs_std_all  = np.zeros((num_diffs, 1))
    diffs_std_pos  = np.zeros((num_diffs, 1))
    diffs_std_neg  = np.zeros((num_diffs, 1))
    diffs_max      = np.zeros((num_diffs, 1))
    diffs_min      = np.zeros((num_diffs, 1))
    datePairs      = [None] * num_diffs

    # Loop through differences to get stats info. 
    for i in range(num_diffs): 
        diffs_mean_all[i] = np.nanmean(da_diffs[i])
        diffs_mean_pos[i] = np.nanmean(da_diffs[i][da_diffs[i]>0.0])
        diffs_mean_neg[i] = np.nanmean(da_diffs[i][da_diffs[i]<0.0])
        
        diffs_med_all[i]  = np.nanmedian(da_diffs[i])
        diffs_med_pos[i]  = np.nanmedian(da_diffs[i][da_diffs[i]>0.0])
        diffs_med_neg[i]  = np.nanmedian(da_diffs[i][da_diffs[i]<0.0])
        
        diffs_std_all[i]  = np.nanstd(da_diffs[i])
        diffs_std_pos[i]  = np.nanstd(da_diffs[i][da_diffs[i]>0.0])
        diffs_std_neg[i]  = np.nanstd(da_diffs[i][da_diffs[i]<0.0])

        diffs_max[i]      = np.nanmax(da_diffs[i])
        diffs_min[i]      = np.nanmin(da_diffs[i])

        # For x axis ticks
        datePairs[i]      = f"{da_tot[i].time.values.astype('datetime64[D]')}\n&\n{da_tot[i+1].time.values.astype('datetime64[D]')}"

    
    # Create figures
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(9,12))

        
    # Plot 'all's
    dummy = np.arange(num_diffs)
    axes[0].plot(datePairs, diffs_mean_all, color='blue')
    axes[0].plot(datePairs, diffs_med_all, color='green')
    axes[0].plot(datePairs, diffs_mean_all+diffs_std_all, color='blue', marker='v', linestyle='None')
    axes[0].plot(datePairs, diffs_mean_all-diffs_std_all, color='blue', marker='^', linestyle='None')
    axes[1].plot(datePairs, diffs_mean_pos, color='blue')
    axes[1].plot(datePairs, diffs_med_pos, color='green')
    axes[1].plot(datePairs, diffs_mean_pos+diffs_std_pos, color='blue', marker='v', linestyle='None')
    axes[1].plot(datePairs, diffs_mean_pos-diffs_std_pos, color='blue', marker='^', linestyle='None')
    axes[2].plot(datePairs, diffs_mean_neg, color='blue')
    axes[2].plot(datePairs, diffs_med_neg, color='green')
    axes[2].plot(datePairs, diffs_mean_neg+diffs_std_neg, color='blue', marker='v', linestyle='None')
    axes[2].plot(datePairs, diffs_mean_neg-diffs_std_neg, color='blue', marker='^', linestyle='None')

    axes[0].grid()
    axes[1].grid()
    axes[2].grid()

    # axes[0].set_xticks(ticks=datePairs, rotation=45)
    axes[0].tick_params(axis='x', labelrotation=45)
    axes[1].tick_params(axis='x', labelrotation=45)
    axes[2].tick_params(axis='x', labelrotation=45)
    
    # Set titles. 
    fig.suptitle("Statistics of Differences\nBlue Line - Mean\nBlue Triangles - Mean +- Standard Deviation\nGreen Line - Median")
    axes[0].set_title("All values")
    axes[1].set_title("Positive values only")
    axes[2].set_title("Negative values only")
    
    # Adjust layout
    plt.tight_layout()

    # Show plot. 
    plt.show()
    
    return

In [ ]:
diffPlotStats(da_diffs_VH[0], da_VH_tot)

In [ ]:
diffPlotStats(da_VH_tot[:-1].values, da_VH_tot)

In [ ]:
diffPlotStats(da_diffs_VV[0], da_VV_tot)

In [ ]:
diffPlotStats(da_VV_tot[:-1].values, da_VV_tot)

In [ ]:
# Histogram plotting. 
def diffPlotHistograms(da_diffs, da_tot, num_bins=30): 

    num_plots = len(da_diffs)

    # Create subplots
    n_cols = 2
    n_rows = int(np.ceil(num_plots/2))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12,n_rows*4))

    # Flatten axes array for easier indexing. 
    axes = axes.flatten()

    myMax = np.nanmax(da_diffs)
    myMin = np.nanmin(da_diffs)

    # Loop through and plot the modes. 
    for i in range(num_plots): 
        ax = axes[i]
        ax.hist(da_diffs[i].flatten(), density=True, log=True, bins=num_bins, range=(myMin, myMax))
        ax.set_title(f"Date Pairs: {da_tot[i].time.values.astype('datetime64[D]')} & {da_tot[i+1].time.values.astype('datetime64[D]')}")
        ax.set_ylabel('Probability')
        ax.set_xlabel('Values')
        ax.grid()

    # Turn off unused subplots
    for i in range(num_plots, len(axes)): 
        ax = axes[i]
        ax.set_axis_off()

    fig.suptitle("Histograms of values")

    # Adjust layout
    plt.tight_layout()

    # Show plot. 
    plt.show()
    
    return

In [ ]:
diffPlotHistograms(da_diffs_VH[0], da_VH_tot, num_bins = 20)

In [ ]:
diffPlotHistograms(da_diffs_VV[0], da_VV_tot, num_bins = 20)

# 5. Find Overflow

Overflow seems to be characterized by a small drop in backscatter following consistent increases. The below code finds possible overflow eventws by searching for positive changes in backscatter (i.e., the backscatter is increasing) followed by a negative change in backscatter (i.e., the backscatter decreases). 

There are three sections:
* 5.1 - Potential Overflow Identification
  * This section finds potential overflow on a pixel by pixel basis.
* 5.2 - Group Pixels
  * This section groups those individual pixels based on two criteria: proximity and time. We group the identified pixels that touch on the same date pair.
* 5.3 - Cull Data Groups
  * This section looks at the mean and median backscatter changes of the grouped pixels and removes those that no longer meet the minimum +/- change expected. 
* 5.4 - Selected Example of Possible Overflow
  * Example plot of the change in backscatter for a possible overflow grouping. 

## 5.1 Potential Overflow Identification

The below code finds which individual pixels meet the criteria for possible overflow. 

In [ ]:
import itertools

In [ ]:
def findOverflowIndices(da_diffs, pos = 0.5, neg = -0.5, start_ind = 0, stop_ind = -1):
    # Function to find indices of overflow. 
    # This method is predicated on overflow being preceded by increasing backscatter, and then decreasing backscatter. 
    # Note: This automatically filters by time. 

    # Input variables:
    # da_diffs - array containing differences. 
    # pos - minimum value for positives. 
    # neg - minimum value for negatives. 

    # Output variables: 
    # myIndices - list of all indices for possible overflow. 
    # idx0 - x indices of myIndices as a list; this can be easier to use than myIndices. 
    # idx1 - y indices of myIndices as a list. 

    # initialize some containers
    list_pos = []
    list_neg = []

    if stop_ind == -1:
        stop_ind = len(da_diffs)-1

    # Loop through all of the difference pairs. 
    for i in range(start_ind, stop_ind):
        # get temporary list of positive "nows" and negative "tomorrows"
        temp_pos = np.where(da_diffs[i] > pos)
        temp_neg = np.where(da_diffs[i+1] < neg)

        # Put positives and their date indices into a list. 
        for j in range(len(temp_pos[0])):
            list_pos.append([temp_pos[0][j], temp_pos[1][j], i])

        # Put negatives and their date indices into a list. 
        for j in range(len(temp_neg[0])):
            list_neg.append([temp_neg[0][j], temp_neg[1][j], i])

    # Put the indices into a single list if they have the same location and time index. 
    myIndices = [list(item) for item in set(map(tuple, list_pos)) & set(map(tuple, list_neg))]
    
    idx0 = [sublist[0] for sublist in myIndices]
    idx1 = [sublist[1] for sublist in myIndices]
    
    return myIndices, idx0, idx1

### <span style="color:red"><b>USER INPUT REQUIRED!!!</b></span>

Please select the minimum changes in backscatter to flag a pixel as a potential location for overflow. 

min_pos - smallest positive change prior to the drop in backscatter. 
min_neg - smallest negative change to backscatter. 

In [ ]:
min_pos = 0.5
min_neg = -3.0

start_ind = 9
stop_ind = -1

In [ ]:
myIndices_VH, idx0_VH, idx1_VH = findOverflowIndices(da_diffs_VH[0], pos=min_pos, neg=min_neg, start_ind=start_ind, stop_ind=stop_ind)
print(f"There are {len(myIndices_VH)} possible overflow pixels identified using VH only.")

In [ ]:
myIndices_VV, idx0_VV, idx1_VV = findOverflowIndices(da_diffs_VV[0], pos=min_pos, neg=min_neg, start_ind=start_ind, stop_ind=stop_ind)
print(f"There are {len(myIndices_VV)} possible overflow pixels identified using VV only.")

In [ ]:
# plot the possible overflow locations from method 1 & 2 on top of each other to highlight major differences

figgy_diffs_VH = plt.figure(figsize=(30,12))
# plot radar image
plt.imshow(da_VH_tot[0], cmap='gray', interpolation='None')

# plot indices as points on radar image. 
plt.plot(idx1_VV, idx0_VV, color='blue', marker='.', markersize=1, linestyle='None', alpha=0.5)
plt.plot(idx1_VH, idx0_VH, color='red', marker='.', markersize=1, linestyle='None', alpha=0.5)

plt.show()

In [ ]:
# plot the possible overflow locations from method 1 & 2 on top of each other to highlight major differences

figgy_diffs = plt.figure(figsize=(30,12))
# plot radar image
plt.imshow(da_VH_tot[0], cmap='gray', interpolation='None')

# plot indices as points on radar image. 
plt.plot(idx1_VH, idx0_VH, color='red', marker='.', markersize=1, linestyle='None', alpha=0.5)

plt.show()

In [ ]:
# plot the possible overflow locations from method 1 & 2 on top of each other to highlight major differences

figgy_diffs_VV = plt.figure(figsize=(30,12))
# plot radar image
plt.imshow(da_VV_tot[0], cmap='gray', interpolation='None')

# plot indices as points on radar image. 
plt.plot(idx1_VV, idx0_VV, color='red', marker='.', markersize=1, linestyle='None', alpha=0.5)

plt.show()

Retain only those indices that appear in both the VV and VH indices. 

In [ ]:
myIndices_both = [list(item) for item in set(map(tuple, myIndices_VH)) & set(map(tuple, myIndices_VV))]
print(f"There are now {len(myIndices_both)} possible overflow pixels identified.")

In [ ]:
idx0_both = [sublist[0] for sublist in myIndices_both]
idx1_both = [sublist[1] for sublist in myIndices_both]

In [ ]:
# plot the possible overflow locations from method 1 & 2 on top of each other to highlight major differences

figgy_diffs_dual = plt.figure(figsize=(30,12))
# plot radar image
plt.imshow(da_VV_tot[0], cmap='gray', interpolation='None')

# plot indices as points on radar image. 
plt.plot(idx1_both, idx0_both, color='red', marker='.', markersize=1, linestyle='None', alpha=0.5)

plt.show()

## 5.2 Group Pixels

The below code finds the pixels identified as potential overflow that touch each other and groups them. Otherwise, we can end up with a list of thousands. 

In [ ]:
# Create a binary mask of overflow pixels

# get shape (size) of array needed. 
ts, ys, xs = da_VH_tot.shape
# initialize an array of zeros. 
# overflow_ungrouped_VH = np.zeros((ys, xs))
# overflow_ungrouped_VV = np.zeros((ys, xs))
overflow_ungrouped_both = np.zeros((ys, xs))
# add 1's in appropriate places. 
# overflow_ungrouped_VH[idx0_VH, idx1_VH] = 1
# overflow_ungrouped_VV[idx0_VV, idx1_VV] = 1
overflow_ungrouped_both[idx0_both, idx1_both] = 1

In [ ]:
# Install skimage
!pip install scikit-image
from skimage.measure import label

In [ ]:
def groupByContact(possible_overflow, da_tot, min_pixels=4, max_pixels=float('inf')):
    # Function to group overflow locations into a dictionary for easy calling/use. 

    # Input variables: 
    # possible_overflow - the mask of potential overflow (0 where no overflow; 1 where possibly detected)
    # da_tot - xarray containing the original SAR image data. 
    # min_pixels - minimum number of pixels a group needs in order to be retained. 

    # Output variables: 
    # myDates - list of dates as strings. 
    # myDatas - raw and statistical data as a dictionary. 

    
    # Group locations by pixels that touch. 
    # connectivity = 1 for only up/down connections. 
    # connectivity = 2 for up/down and diagonal connections. 
    labeled_image = label(possible_overflow, connectivity=2)

    # Remove groups of pixels that are smaller than 'min_pixels.' 
    for i in range(1,np.max(labeled_image)+1): 
        pixelIndices = np.where(labeled_image == i)
        pixelCount = len(pixelIndices[0])
        if (pixelCount < min_pixels) or (pixelCount > max_pixels): 
            labeled_image[np.where(labeled_image == i)] = 0

    # Put raw data into dictionary for each grouping. 
    # Get unique entries and their counts. Used for organizing later. 
    unique_vals, counts = np.unique(labeled_image, return_counts=True)    
    # initialize container for backscatter values. 
    myDatas = {}    
    myIndices = {}
    for val in unique_vals[1:]: # skipping 0 since that will just be non-overflow. 
        myIndex = np.where(labeled_image == val)
        ts = len(da_tot)
        ps = len(myIndex[0])
        myDates = da_tot[:].time.values.astype('datetime64[D]')
        myValues = np.zeros(( ts, ps))
        # Add values to dictionary. 
        myData = {}
        for i in range(ps):
            myValues[:, i] = da_tot[:, myIndex[0][i], myIndex[1][i]].values
        myData['raw'] = myValues
        myDatas[f"{val}"] = myData
        # Add indices of location to dictionary. 
        myIndices = []
        for i in range(len(myIndex[0])):
            myIndices.append([myIndex[0][i], myIndex[1][i]])
        myDatas[f"{val}"]['indices'] = myIndices

    # Get statistics into 'myDatas.' 
    for key in myDatas:
        # Initialize containers for mean, median, and std. 
        myMeans   = np.zeros((1,0))
        myMedians = np.zeros((1,0))
        myStds    = np.zeros((1,0))
        # Get stats for each key. 
        for i in range(len(myDatas[key]['raw'])):
            myMean =  np.nanmean(myDatas[key]['raw'][i])
            myMeans = np.append(myMeans,myMean)
    
            myMedian =  np.nanmedian(myDatas[key]['raw'][i])
            myMedians = np.append(myMedians,myMedian)
    
            myStd =  np.nanstd(myDatas[key]['raw'][i])
            myStds = np.append(myStds, myStd)
    
        # Put stats in dictionary. 
        myDatas[key]['mean']   = myMeans
        myDatas[key]['median'] = myMedians
        myDatas[key]['std']    = myStds
    
    return myDatas, myDates

### <span style="color:red"><b>USER INPUT REQUIRED!!!</b></span>

Please select the minimum number of touching pixels (min_pixels) for your overflow region. Note that the maximum may also be selected. 

In [ ]:
# Typical overflow event are approximately 15 S1 pixels in size. 
min_pixels = 10
max_pixels = float('inf')

In [ ]:
myDatas_VH, myDates = groupByContact(overflow_ungrouped_both, 
                                     da_VH_tot, 
                                     min_pixels=min_pixels,
                                     max_pixels=max_pixels)

myDatas_VV, myDates = groupByContact(overflow_ungrouped_both, 
                                     da_VV_tot, 
                                     min_pixels=min_pixels,
                                     max_pixels=max_pixels)

In [ ]:
# Get an idea of how large the locations of possible overflow are. 
for key in myDatas_VH:
    print(f"Key: {key},  # of indices: {len(myDatas_VH[key]['indices'])}")

In [ ]:
# Get an idea of how large the locations of possible overflow are. 
for key in myDatas_VV:
    print(f"Key: {key},  # of indices: {len(myDatas_VV[key]['indices'])}")

In [ ]:
# Plot data for user to look through graphs and identify likely days/locations with overflow. 

def plotdBForEachLocation(myDatas, myDates, ymin=-24.0, ymax=-5.0):

    # loop through each key in 'myDatas' dictionary. 
    # Note: The keys should correspond to the unique values of possible overflow groupings. 
    
    ncols = 2
    nrows = int(np.ceil(len(myDatas)/2))
    fig_find, axes_find = plt.subplots(nrows=nrows, ncols=ncols, figsize=(9,nrows*4))
    
    axes_find = axes_find.flatten()
    
    for i, key in enumerate(myDatas): 
        ax = axes_find[i]
        ax.plot(myDates, myDatas[key]['mean'], color='blue')
        ax.plot(myDates, myDatas[key]['mean']+myDatas[key]['std'], color='blue', marker='v', linestyle='None')
        ax.plot(myDates, myDatas[key]['mean']-myDatas[key]['std'], color='blue', marker='^', linestyle='None')
        ax.plot(myDates, myDatas[key]['median'], color='green')
        ax.grid()    
        ax.set_xticks(myDates, myDates, rotation=45)
        ax.set_title(f"Location Number: {key}")
        ax.set_ylim(ymin, ymax)

    for j in range(i+1, len(axes_find)):
        ax = axes_find[j]
        ax.set_axis_off()
    
    plt.tight_layout()
    plt.show()

### <span style="color:red"><b>NOTE: </b></span>

The graphs below will be used to select which of the possible overflow locations to examine in further detail. Typically, overflow is preceded by slowly increasing backscatter through time with a minor spike in the data before a small (but noticeable) drop in backscatter. 

Typically, locations likely to have overflow have background backscatter values greater than -22 dB. 

In [ ]:
plotdBForEachLocation(myDatas_VH, myDates, ymin = -24., ymax = -14.)

In [ ]:
plotdBForEachLocation(myDatas_VV, myDates, ymin = -24., ymax = -5.)

## 5.3 Plot grouped Locations

Plot the grouped locations with annotations.

In [ ]:
# plot the grouped locations and export as a geotiff. 

figgy_annotated = plt.figure(figsize=(30,12))
# plot radar image
plt.imshow(da_VV_tot[0], cmap='gray', interpolation='None')

# plot indices as points on radar image. 
for key in myDatas_VV:
    idx1_temp, idx0_temp = [], []
    for i in range(len(myDatas_VV[key]['indices'])):
        idx1_temp.append(myDatas_VV[key]['indices'][i][1])
        idx0_temp.append(myDatas_VV[key]['indices'][i][0])
    plt.plot(idx1_temp, idx0_temp, color='red', marker='.', markersize=1, linestyle='None', alpha=0.5)
    plt.annotate(f"Location: {key}", xy=(int(np.min(idx1_temp))+10, int(np.max(idx0_temp)-10)))

plt.show()

Export locations as a kml file for viewing in a GIS. 

In [ ]:
from pyproj import Transformer

def getLatLon(parent, myData_indices):
    # Function to get latitude and longitude from a geotiff for particular indices. 

    # Input variables: 
    # parent - posix path of the parent directory. 
    # myData_indices - list of lists of the indices. 

    # Output variables: 
    # lat - latitude in degrees. 
    # lon - longitude in degrees. 

    # Get the center index from the list of indices. 
    idx0, idx1 = [], []
    for i in range(len(myData_indices)):
        idx0.append(myData_indices[i][0])
        idx1.append(myData_indices[i][1])
    row = int( np.median( idx0 ) )
    col = int( np.median( idx1 ) )

    # Get a tiff path. 
    folder = parent+'RTC_GAMMA/'
    # Create the path to the tiff directory. 
    tiff_dir = Path(folder)
    # Get all the tiff names from the tiff directory. 
    tiffs = [f for f in os.listdir(tiff_dir)]
    # Put together folder and one tiff into a complete path. 
    myTiffPath = Path(folder, tiffs[0])
    # Open the tiff in order to get the lat/lon. 
    with rasterio.open(myTiffPath) as src: 
        # Get the x and y coordinate from our chosen index. 
        x, y = src.transform * (col, row)
        # Get the coordinate reference system. 
        crs = src.crs
        # Setup object to convert x and y into lat/lon. 
        transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        # convert x/y to lat/lon. 
        lon, lat = transformer.transform(x,y)
    
    
    return lat, lon

Select folder in which to save the kml. 

In [ ]:
fc_saveLoc = FileChooser(Path.cwd())
display(fc_saveLoc)

In [ ]:
!pip install simplekml

In [ ]:

import simplekml
myKml = simplekml.Kml()

for key in myDatas_VV:
    lat, lon = getLatLon(fc.selected, myDatas_VH[key]['indices'])
    # print(f"Lat/Lon: {lat}, {lon}")
    myKml.newpoint(name=f"Location {key}", coords=[(lon, lat)])

myKml_filename = Path(fc_saveLoc.selected,"possible-S1-overflow.kml")

myKml.save(myKml_filename)

print(f"KML file saved as: {myKml_filename}")

### 5.4 Selected example of a possible overflow event. 

The below cell plots a possible overflow event from the example data. The possible overflow occurs between <b>2024-11-24</b> and <b>2024-12-06</b>. 

In [ ]:
fig_pof = plt.figure()

ymin = -24.0
ymax = -14.0

loc = '33'
plt.plot(myDates, myDatas_v1[loc]['mean'], color='blue')
plt.plot(myDates, myDatas_v1[loc]['mean']+myDatas_v1[loc]['std'], color='blue', marker='v', linestyle='None')
plt.plot(myDates, myDatas_v1[loc]['mean']-myDatas_v1[loc]['std'], color='blue', marker='^', linestyle='None')
plt.plot(myDates, myDatas_v1[loc]['median'], color='green')
plt.grid()    
plt.xticks(myDates, myDates, rotation=45)
plt.title(f"Location Number: {loc}")
plt.ylim(ymin, ymax)

plt.tight_layout()
plt.show()

The possible overflow between <b>2024-11-24</b> and <b>2024-12-06</b> shows a typical spike in backscatter followed by a minor decrease. 

# 6. Display Data for Possible Overflow Locations

In [ ]:
# plot backscatter and diffs through time for identified indices. 
def plotBSnDiffsThruTime(da_diffs, da_tot, myIndices):
    num_mI = len(myIndices)
    
    # Create subplots. 
    n_cols = 2
    n_rows = int(np.ceil(num_mI))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, n_rows*4))
    
    # Flatten axes array for easier indexing. 
    axes = axes.flatten()
    
    myDates = da_VH_tot.time.values.astype('datetime64[D]')
    myDatePairs = [None] * (len(myDates)-1)
    for i in range(len(myDatePairs)):
        myDatePairs[i] = f"{myDates[i]} & {myDates[i+1]}"
    
    for j in range(0,2*num_mI, 2): 
        i = int(j/2)
        # Pull out data for each possible location to a temporary variable for plotting. 
        myData_raw   = da_tot[:, myIndices[i][0], myIndices[i][1]].values
        myData_diffs = da_diffs[:, myIndices[i][0], myIndices[i][1]]
        # Create a dummy variable. 
        dummy_raw = np.arange(len(myData_raw))
        dummy_diffs = np.arange(len(myData_diffs))
        # Plot the raw data. 
        ax_raw = axes[j]
        ax_raw.plot(myDates, myData_raw)
        # Plot the difference data. 
        ax_diff = axes[j+1]
        ax_diff.plot(myDatePairs, myData_diffs)
    
        # Format plots
        ax_raw.grid()
        ax_diff.grid()
        ax_raw.set_title(f"Raw Data [dB]")
        ax_diff.set_title(f"Differences")
        ax_raw.set_ylabel("Decibels")
        ax_raw.tick_params(axis='x', labelrotation=45)
        ax_diff.tick_params(axis='x', labelrotation=45)
        
    
    plt.tight_layout()
    plt.show()

    return

### <span style="color:red"><b>USER INPUT REQUIRED!!!</b></span>

Please select the location number for your overflow region (loc). 

In [ ]:
loc = '5'
myDate = np.datetime64('2026-01-24')

In [ ]:
# Display raw backscatter and change in backscatter pairs for all indices in the given location. 
# I don't consider this particularly useful; I include it so users may examine each pixel 
# within a group for strange behavior. 
plotBSnDiffsThruTime(da_diffs_VH[0], da_VH_tot, myDatas_VH[loc]['indices'])

In [ ]:
# plot locations for identified indices as pairs (pre-overflow and post-overflow). 

def plotPossOverflowLocations(da, myIndices, DOI, w_s, factor = 30, vmin = -25.0, vmax = -5.0): 

    # plot locations for identified indices as pairs (pre-overflow and post-overflow). 

    # da - input data; xarray expected. 
    # myIndices - identified overflow indices. 
    # DOI - date of interest. Needs to be 
    # w_s - window size; scalar expected. 
    # factor - scale by which to multiple w_s for display; scalar expected. 
    # vmin - minimum values to plot; default is scaled for dB input. 
    # vmax - maximum values to plot; default is scaled for dB input. 

    num_mI = len(myIndices)
    w_s = int(np.floor(w_s/2))
        
    # Create subplots. 
    n_cols = 2
    n_rows = 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, n_rows*4))
    
    # Flatten axes array for easier indexing. 
    axes    = axes.flatten()
    ax_pre  = axes[0]
    ax_post = axes[1]
    
    myDates = da.time.values.astype('datetime64[D]')
    dateIndex = np.where(myDates == DOI)[0][0]
    myDatePair = f"{myDates[dateIndex]} & {myDates[dateIndex+1]}"


    # Pull out data for each possible location to a temporary variable for plotting. 
    idx0 = []
    idx1 = []
    for i in range(len(myIndices)): 
        idx0.append(myIndices[i][0])
        idx1.append(myIndices[i][1])
    idx0 = int(np.median(idx0))
    idx1 = int(np.median(idx1))

    # Get extents to plot
    xmin, xmax = idx1-factor*w_s, idx1+factor*w_s+1
    ymin, ymax = idx0-factor*w_s, idx0+factor*w_s+1
    if xmin < 0: xmin = 0
    if xmax > da.shape[2]: xmax = da.shape[2]
    if ymin < 0: ymin = 0
    if ymax > da.shape[1]: ymax = da.shape[1]

    # plot the pre- and post-overflow data. 
    ax_pre.imshow(da[dateIndex], cmap='gray', interpolation=None, vmin=vmin, vmax=vmax)
    ax_post.imshow(da[dateIndex+1], cmap='gray', interpolation=None, vmin=vmin, vmax=vmax)
    # Constrain the view to the location of interest. 
    ax_pre.set_xlim(xmin, xmax)
    ax_pre.set_ylim(ymin, ymax)
    ax_post.set_xlim(xmin, xmax)
    ax_post.set_ylim(ymin, ymax)
    # Plot the position of the point of change. 
    ax_pre.plot(idx1, idx0, marker='o', markersize=20, fillstyle='none', linestyle='None')
    ax_post.plot(idx1, idx0, marker='o', markersize=20, fillstyle='none', linestyle='None')
    # Correct y axis flip; using imshow and then .plot will flip the y axis so north is down. 
    ax_pre.invert_yaxis()
    ax_post.invert_yaxis()

    # Format plots
    ax_pre.grid()
    ax_post.grid()
    ax_pre.set_title(f"Pre-Overflow\nDate: {myDates[dateIndex]}")
    ax_post.set_title(f"Post-Overflow\nDate: {myDates[dateIndex+1]}")
        
    
    plt.tight_layout()
    plt.show()

    return

### <span style="color:red"><b>USER INPUT REQUIRED!!!</b></span>

Please select the date that directly precedes the possible overflow detection(myDate). 

In [ ]:
loc = '116'
myDate = np.datetime64('2025-08-09')

In [ ]:
# Display the pre- and post-overflow SAR image for a given date. 
# plotPossOverflowLocations(da_VH_tot, myDatas_VH[loc]['indices'], myDate, window_sizes[0], factor=3)
plotPossOverflowLocations(da_VH_tot, myDatas_VH[loc]['indices'], myDate, window_sizes[0], factor=20)
plotPossOverflowLocations(da_VV_tot, myDatas_VV[loc]['indices'], myDate, window_sizes[0], factor=20)

In [ ]:
# Display all of the SAR image pairs for a given location. 

for myDate in myDates[0:-1]:
    print(f'Set for {myDate}')
    # plotPossOverflowLocations(da_VH_tot, myDatas_VH[loc]['indices'], myDate, window_sizes[0], factor=3)
    plotPossOverflowLocations(da_VH_tot, myDatas_VH[loc]['indices'], myDate, window_sizes[0], factor=10)
    print(f'\n\n\n')

In [ ]:
# Display all of the SAR image pairs for a given location. 

for myDate in myDates[0:-1]:
    print(f'Set for {myDate}')
    # plotPossOverflowLocations(da_VV_tot, myDatas_VV[loc]['indices'], myDate, window_sizes[0], factor=3)
    plotPossOverflowLocations(da_VV_tot, myDatas_VV[loc]['indices'], myDate, window_sizes[0], factor=10)
    print(f'\n\n\n')

# 7. Extract geographic location of selected overflow event

The below code will extract the latitude and longitude of a single user-selected overflow event. 

In [ ]:
from pyproj import Transformer

def getLatLon(parent, myData_indices):
    # Function to get latitude and longitude from a geotiff for particular indices. 

    # Input variables: 
    # parent - posix path of the parent directory. 
    # myData_indices - list of lists of the indices. 

    # Output variables: 
    # lat - latitude in degrees. 
    # lon - longitude in degrees. 

    # Get the center index from the list of indices. 
    idx0, idx1 = [], []
    for i in range(len(myData_indices)):
        idx0.append(myData_indices[i][0])
        idx1.append(myData_indices[i][1])
    row = int( np.median( idx0 ) )
    col = int( np.median( idx1 ) )

    # Get a tiff path. 
    folder = parent+'RTC_GAMMA/'
    # Create the path to the tiff directory. 
    tiff_dir = Path(folder)
    # Get all the tiff names from the tiff directory. 
    tiffs = [f for f in os.listdir(tiff_dir)]
    # Put together folder and one tiff into a complete path. 
    myTiffPath = Path(folder, tiffs[0])
    # Open the tiff in order to get the lat/lon. 
    with rasterio.open(myTiffPath) as src: 
        # Get the x and y coordinate from our chosen index. 
        x, y = src.transform * (col, row)
        # Get the coordinate reference system. 
        crs = src.crs
        # Setup object to convert x and y into lat/lon. 
        transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        # convert x/y to lat/lon. 
        lon, lat = transformer.transform(x,y)
    
    
    return lat, lon

In [ ]:
lat, lon = getLatLon(fc.selected, myDatas_VH[loc]['indices'])
print(f"Lat: {lat}")
print(f"Lon: {lon}")